# Codify 301: compare legal documents

Compare a reference instrument with another document. For each reference provision,
Codify retrieves relevant text, asks a model to assess alignment, and records a
finding with citations and confidence. Findings need legal review.

Use the setup in [Codify 101](codify-101.ipynb) with chat and embeddings endpoints.
This example reads files and does not need a database.

The comparison makes embedding requests and chat calls for provisions it assesses.
Validation failures may trigger retries. Cost depends on the model and documents.

In [1]:
import logging
from pathlib import Path

import langfuse  # noqa: F401, its import resets the logger quieted below
import structlog
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))  # the repo's .env; exported variables win
logging.getLogger("langfuse").setLevel(logging.ERROR)  # tracing is optional
structlog.configure(  # warnings only, uncoloured
    wrapper_class=structlog.make_filtering_bound_logger(logging.WARNING),
    processors=[structlog.processors.add_log_level, structlog.dev.ConsoleRenderer(colors=False)],
)
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
FIXTURES = REPO / "tests" / "fixtures" / "synthetic"

## 1. Load the documents

Both documents come from the fictional Hesperian Union (`xu`). A data-protection
directive is the reference. A later regulation establishing a data-protection board
is the document being assessed. It covers only part of the directive, so this
example should identify many gaps.

Pass additional domestic documents through `corpus=` when the relevant law spans
several instruments. A gap in this small example does not establish a gap in an
entire legal system.

In [2]:
import json

from codify.akn import parse_akn
from codify.akn.bluebell import bluebell_to_akn

laws = json.loads((FIXTURES / "xu" / "manifest.json").read_text())["laws"]
manifest = {law["file"]: law for law in laws}


def load(name):
    law = manifest[name]
    akn_xml, _, errors = bluebell_to_akn(
        (FIXTURES / "xu" / name).read_text(),
        country="xu",
        doctype=law["doctype"],
        date=law["date"][:4],
        number=law["number"],
    )
    assert not errors, errors
    return parse_akn(akn_xml)


directive = load("directive-2004-12-data-protection.bluebell")
domestic = load("regulation-2009-5-data-protection-board.bluebell")
print(directive.frbr_work_uri, "->", domestic.frbr_work_uri)

/akn/xu/act/2004/12 -> /akn/xu/act/2009/5


`iter_assessable` lists the reference units in document order, including structural
units that do not need a model call. The assessment focuses on provisions that
state obligations.

In [3]:
from codify.compare.scaffold import iter_assessable

units = list(iter_assessable(directive))
print(f"{len(units)} assessable provisions in the directive")
for u in units[:6]:
    print(f"  {u.akn_eid:<22} {(u.heading or u.text)[:70]}")

55 assessable provisions in the directive
  chp_1__art_1           Subject matter and objectives
  chp_1__art_1__para_1   This Directive lays down rules relating to the protection of natural p
  chp_1__art_1__para_2   This Directive protects the fundamental rights and freedoms of natural
  chp_1__art_1__para_3   The free movement of personal data within the Union shall be neither r
  chp_1__art_2           Material scope
  chp_1__art_2__para_1   This Directive applies to the processing of personal data wholly or pa


## 2. Run the comparison

`llm_concurrency` limits concurrent model calls. If you use `seed`, check whether
your endpoint supports it. It does not guarantee identical results.

In [4]:
import os

from codify.compare.comparator import compare
from codify.core.llm import create_llm_client
from codify.embed.client import EmbeddingClient

llm = create_llm_client(
    base_url=os.environ["LITELLM_BASE_URL"],
    api_key=os.environ["LITELLM_API_KEY"],
    model=os.environ.get("LITELLM_MODEL", "gemini-3.7-flash"),
    telemetry_mode="direct",
)
embedder = EmbeddingClient(
    base_url=os.environ.get("EMBEDDING_BASE_URL") or os.environ["LITELLM_BASE_URL"],
    api_key=os.environ.get("EMBEDDING_API_KEY") or os.environ["LITELLM_API_KEY"],
    model=os.environ.get("EMBEDDING_MODEL", "gemini-embedding-2"),
)

report = await compare(
    directive,
    domestic,
    llm=llm,
    embedding_client=embedder,
    llm_concurrency=4,
    on_progress=lambda ev: print(f"  {ev.done}/{ev.total}") if ev.done % 10 == 0 else None,
)
print(report.summary.model_dump())

  10/55


  20/55


  30/55


  40/55


  50/55


{'aligned': 1, 'partial': 0, 'gap': 31, 'total': 32, 'aligned_pct': 3.125, 'partial_pct': 0.0, 'gap_pct': 96.875, 'needs_review': 0, 'na': 23, 'key_gaps': ['Subject matter and objectives', 'Material scope', 'Applicable law', 'Principles relating to processing', 'Lawfulness of processing', 'Special categories of personal data', 'Information to be provided to the data subject', 'Right of access']}


## 3. Read the report

Each row describes a reference provision. Rows with `actionable=false` do not
count towards the alignment summary and appear as `na`.

`needs_review` marks findings below the confidence threshold. Review the cited text
as well as the finding; model confidence is not a measure of legal correctness.

In [5]:
from collections import Counter

actionable = [r for r in report.results if r.actionable]
print(Counter(r.verdict for r in actionable))
print(len(report.results) - len(actionable), "not actionable")
print()
for r in actionable:
    cites = ", ".join(c.akn_eid for c in r.citations) or "-"
    flag = " review" if r.needs_review else ""
    print(f"{r.directive_eid:<22} {r.verdict:<8} {r.confidence:.2f}  {cites:<30}{flag}")

Counter({'gap': 31, 'aligned': 1})
23 not actionable

chp_1__art_1__para_3   gap      0.95  -                             
chp_1__art_2__para_1   gap      0.95  -                             
chp_1__art_4__para_1   gap      0.95  -                             
chp_1__art_4__para_2   gap      0.95  -                             
chp_2__art_5__para_1   gap      0.95  -                             
chp_2__art_5__para_2   gap      1.00  -                             
chp_2__art_6__para_1   gap      0.98  -                             
chp_2__art_6__para_2   gap      0.95  -                             
chp_2__art_7__para_1   gap      0.98  -                             
chp_2__art_7__para_2   gap      0.95  -                             
chp_2__art_8__para_1   gap      0.95  -                             
chp_2__art_8__para_2   gap      0.95  -                             
chp_2__art_9           gap      0.95  -                             
chp_2__art_10__para_1  gap      1.00  -          

The examples below show an alignment finding and a gap, with the model's
explanation and any domestic provision it cites.

In [6]:
by_verdict = {}
for r in actionable:
    by_verdict.setdefault(r.verdict, r)
for verdict in ("aligned", "partial", "gap"):
    r = by_verdict.get(verdict)
    if r is None:
        continue
    print(f"[{verdict}] {r.directive_eid}: {r.directive_heading}")
    for c in r.citations:
        print(f"   cites {c.akn_eid}  {c.frbr_uri}")
    print(f"   {r.note}\n")

[aligned] chp_3__art_17__para_2: Supervisory authority
   cites chp_1__art_4__para_1  /akn/xu/act/2009/5/eng
   cites chp_1__art_4__para_2  /akn/xu/act/2009/5/eng
   The domestic law transposes the independence requirement directly, providing that the authority shall act with complete independence and remain free from external instructions or influence.

[gap] chp_1__art_1__para_3: Subject matter and objectives
   The directive provision establishes a fundamental obligation that the free movement of personal data shall not be restricted or prohibited on data protection grounds. None of the retrieved candidate provisions address the free movement of personal data or prohibit restrictions on data flows.



## 4. Compare stored versions

After loading documents as in [Codify 201](codify-201.ipynb), compare them by
version ID. This lets Codify cache their embeddings:

```bash
uv run codify compare <reference-version-id> <domestic-version-id> --out report.json
```

The MCP equivalent is `compare_versions(reference_version_id, domestic_version_id)`.
By default, it accepts at most 200 articles, sections and paragraphs after excluding
structural units and units excluded by the document’s rules (such as placeholders).
Points do not count separately.

Use `report.model_dump_json(indent=2)` to export the Python report as JSON.

In [7]:
print(report.model_dump_json(indent=2, exclude={"generated_at"})[:900])

{
  "directive_frbr_uri": "/akn/xu/act/2004/12/eng",
  "domestic_frbr_uri": "/akn/xu/act/2009/5/eng",
  "seed": null,
  "confidence_threshold": 0.7,
  "results": [
    {
      "directive_eid": "chp_1__art_1",
      "directive_heading": "Subject matter and objectives",
      "verdict": "gap",
      "confidence": 1.0,
      "note": "Structural heading: no transposition obligation.",
      "citations": [],
      "needs_review": false,
      "actionable": false,
      "provision_kind": "structural",
      "clause_method": "na",
      "rejected_candidates": [],
      "langfuse_trace_id": null,
      "trace_url": null
    },
    {
      "directive_eid": "chp_1__art_1__para_1",
      "directive_heading": "Subject matter and objectives",
      "verdict": "aligned",
      "confidence": 0.95,
      "note": "The directive provision is a non-actionable introductory statement of subject matter and sc
